# Single ages and custom age bands — Track B — Census-anchored

The population series are built in 16 five-year bands. Models and policy analyses often need other groupings — for example
**0–8, 9–14, 15–19** followed by five-year bands (a target age group for an intervention), or single years of age.
This notebook splits the five-year bands of **Track B** into single ages 0 … 99 and 100+, then regroups them into
custom bands, and compares four ways of doing the split.

## Methods

All four reproduce every five-year band total exactly; they differ in how the total is spread over the ages inside the band.

| Method | How ages inside a band are filled | Uses WPP single ages | Smooth across band edges |
|---|---|---|---|
| **M1 uniform** | Equal share for each age (75+ spread equally over 75 … 100+) | no | no |
| **M2 WPP proportions** | Each age gets WPP's share of the band in the same year | yes | no |
| **M3 cumulative spline** | Monotone cubic curve (PCHIP) through the cumulative population at the band edges | no | yes |
| **M4 WPP-guided smooth** | WPP single ages × a correction that is continuous and piecewise linear in age, solved so every band total is exact | yes | yes |

**M4 in formulas.** With $W_a(t)$ the WPP single-age population and $B_j(a)$ piecewise-linear "hat" functions centred on the band
mid-points, the single-age population is
$$P_a(t) = W_a(t)\,\rho(a,t),\qquad \rho(a,t) = \sum_{j=1}^{16} v_j(t)\,B_j(a),$$
and the 16 coefficients $v_j(t)$ solve the linear system $\sum_{a\in b} W_a(t)\,\rho(a,t) = P_b(t)$ for every band $b$. This is the
single-age version of the band-wise ratio method used for Track B: WPP supplies the shape, the track supplies the levels.

## What is tested

1. **Benchmark with a known answer** (identical in the three track notebooks): WPP's own single ages are collapsed into bands and
   split back, using WPP's single-age shape from a *different* year to mimic a population that does not match WPP exactly.
2. **Single ages from this track**: age profiles, steps at band edges, and a one-year cohort test.
3. **Custom bands**: several grouping schemes, checked for new kinks over time and compared across methods.
4. **States**: how much the method matters for a policy band (9–14) in each state.

## 0. Settings and data

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pathlib, os, warnings
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))   # the repository's src/ when run from notebooks/
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import popproj as px
from popproj import sp
import popmodel as pm
import popsingle as ps
warnings.filterwarnings("ignore", category=RuntimeWarning)

TRACK   = "B"
VARIANT = "B-taper"          # options: `B-taper`, `B-hold`, `B-smooth`
TAG = f"track{TRACK}_{VARIANT}"
px.configure(fig_dir=os.path.join(px.REPO_ROOT, "figures", "single_age", TAG))
px.setup_style(scale=0.75)
pd.set_option("display.width", 200, "display.max_columns", 40)

W = ps.wpp_single()                                        # WPP single ages, years x 101
india = pm.load_track(TRACK, VARIANT)                      # years x 16 bands
_long = pd.read_parquet(os.path.join(pm.OUT_DIR, f"track{TRACK}", f"track{TRACK}_states_{pm.DEFAULT_STATE_VARIANT[TRACK]}.parquet"))
STATES = sorted(_long.state.unique())
states = {s: _long[_long.state == s].pivot(index="year", columns="band", values="females")[ps.BANDS] for s in STATES}
MCOL = {"uniform": sp.GREY, "wpp": sp.L2, "cumspline": sp.ACC, "wpp_smooth": sp.OUTC}
MSTY = {"uniform": ":", "wpp": "--", "cumspline": "-.", "wpp_smooth": "-"}
print(f"Track {TRACK} ({VARIANT}): India {india.shape}, {len(states)} states; WPP single ages {W.shape}")

---
## 1. Benchmark: splitting WPP back into its own single ages

For each year $t$ (1960–2080), WPP's single ages are summed into the 16 bands and split back. M2 and M4 are given WPP's single-age
shape from year $t+\Delta$ instead of $t$ — with $\Delta = 0$ they have the exact shape (and are exact); with $\Delta \ne 0$ the shape
is wrong in the way a state's or track's shape can differ from WPP. M1 and M3 do not use the shape.

In [ ]:
bench = ps.benchmark(W)
fig, axs = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, col in zip(axs, ["MAPE ages 0-74 %", "MAPE ages 75-99 %", "max APE 0-99 %"]):
    for m in ps.METHODS:
        d = bench[bench.method == ps.METHOD_LABEL[m]].sort_values("shape shift (yr)")
        ax.plot(d["shape shift (yr)"], d[col].clip(lower=1e-3), marker="o", color=MCOL[m], ls=MSTY[m], label=ps.METHOD_LABEL[m])
    ax.set_yscale("log"); ax.set_xlabel("Shift of the WPP shape Δ [years]"); ax.set_ylabel(col.replace(" %", " [%]"))
    ax.set_title(col.split(":")[0]); ax.grid(alpha=0.25)
axs[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); px.save(fig, f"{TAG}_benchmark"); plt.show()

# single-age profile, 2011 split with the 2031 shape (Δ = +20)
t, dt = 2011, 20
truth = W.loc[[t]]
bands = pd.DataFrame(truth.values @ ps.MEMBER.T, index=[t], columns=ps.BANDS)
shape = pd.DataFrame(W.loc[[t + dt]].values, index=[t], columns=ps.AGES)
fig, axs = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, (lo, hi) in zip(axs, [(0, 30), (40, 75), (70, 100)]):
    a = np.arange(lo, hi + 1)
    ax.plot(a, truth.loc[t, lo:hi] / 1e6, color="black", lw=2.2, label="WPP truth")
    for m in ps.METHODS:
        ax.plot(a, ps.split(bands, m, shape).loc[t, lo:hi] / 1e6, color=MCOL[m], ls=MSTY[m], lw=1.5, label=ps.METHOD_LABEL[m])
    for lo_b, _ in ps.BAND_RANGE:
        if lo < lo_b <= hi: ax.axvline(lo_b - 0.5, color=sp.GREY, lw=0.5, ls=":")
    ax.set_xlabel("Age [years]"); ax.set_ylabel("Female pop. [millions]"); ax.set_title(f"Ages {lo}-{hi}"); ax.grid(alpha=0.25)
axs[0].legend(frameon=False, fontsize=8)
fig.suptitle(f"{t} split with the WPP shape of {t + dt} (dotted lines = band edges)", y=1.03)
plt.tight_layout(); px.save(fig, f"{TAG}_benchmark_profile"); plt.show()
bench.round(2)

---
## 2. Single ages from this track (India)

In [ ]:
S = {m: ps.split(india, m, W) for m in ps.METHODS}                 # method -> years x 101
chk = max((ps.regroup(S[m], ps.SCHEMES["S0 original 5-year bands"]).values / india.values - 1).max() for m in ps.METHODS)
print(f"Regrouping every method back to the 16 bands reproduces the track: max relative difference {chk:.1e}")

### 2.1 Single-age profiles

Dotted vertical lines mark the edges of the five-year bands. Steps at these lines are artefacts of the method.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(16, 9))
for ax, yr in zip(axs.flat, [1950, 2011, 2036, 2100]):
    for m in ps.METHODS:
        ax.plot(ps.AGES, S[m].loc[yr] / 1e6, color=MCOL[m], ls=MSTY[m], lw=1.4, label=ps.METHOD_LABEL[m])
    for lo_b, _ in ps.BAND_RANGE[1:]:
        ax.axvline(lo_b - 0.5, color=sp.GREY, lw=0.5, ls=":")
    ax.set_xlabel("Age [years]"); ax.set_ylabel("Female pop. [millions]"); ax.set_title(f"{yr}"); ax.grid(alpha=0.25)
axs[0, 0].legend(frameon=False, fontsize=8)
plt.tight_layout(); px.save(fig, f"{TAG}_profiles"); plt.show()

### 2.2 Steps at band edges

For each age, how much the age profile bends: $|g(a) - \tfrac12(g(a-1)+g(a+1))|$ with $g(a)=\ln(P_{a+1}/P_a)$, averaged at band edges and
inside bands (ages 1–73) over all years. A smooth split has similar values in both; a method that creates steps has much larger
values at the edges.

In [ ]:
es = pd.DataFrame({ps.METHOD_LABEL[m]: ps.edge_steps(S[m]).mean() for m in ps.METHODS}).T
es["edge / inside"] = es["at band edges"] / es["inside bands"]
fig, ax = plt.subplots(figsize=(9, 4.2))
x = np.arange(len(es))
ax.bar(x - 0.2, es["at band edges"], width=0.4, color=sp.OUTC, label="at band edges")
ax.bar(x + 0.2, es["inside bands"], width=0.4, color=sp.L2, label="inside bands")
ax.set_xticks(x); ax.set_xticklabels(es.index); sp.lock_ticks(ax, "x")
ax.set_ylabel("Mean bend of the age profile"); ax.set_title("Band-edge steps (lower = smoother)"); ax.grid(alpha=0.25, axis="y")
ax.legend(frameon=False)
plt.tight_layout(); px.save(fig, f"{TAG}_edge_steps"); plt.show()
es.round(4)

### 2.3 One-year cohort test

$P_{a+1}(t+1)/P_a(t)$ for ages 0–98: the fraction of women aged $a$ who are still present one year later at age $a+1$. Without
migration it cannot exceed 1. Red cells (> 1) mean the single-age series makes a cohort grow — either inherited from the track
itself or created by the split (typically at band edges).

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
norm = TwoSlopeNorm(vmin=-0.05, vcenter=0, vmax=0.05)
grow = {}
for ax, m in zip(axs, ps.METHODS):
    c = np.log(ps.cohort_1y(S[m]))
    grow[ps.METHOD_LABEL[m]] = {"cells with cohort growth %": (c.values > 0).mean() * 100,
                                "at band edges %": (c.loc[:, [hi for _, hi in ps.BAND_RANGE[:-2]]].values > 0).mean() * 100}
    im = ax.imshow(c.T.values, aspect="auto", cmap="RdBu_r", norm=norm, origin="lower",
                   extent=[c.index.min() - 0.5, c.index.max() + 0.5, -0.5, 98.5])
    ax.set_title(ps.METHOD_LABEL[m]); ax.set_xlabel("Year"); ax.set_ylabel("Age a [years]")
fig.colorbar(im, ax=axs, shrink=0.85, label="log one-year survival (red = cohort grows)")
px.save(fig, f"{TAG}_cohort_1y"); plt.show()
pd.DataFrame(grow).T.round(1)

### 2.4 Single ages over time

Ages on both sides of band edges (9 | 10, 14 | 15, 19 | 20) and inside a band (12, 17).

In [ ]:
AG = [9, 10, 12, 14, 15, 17, 19, 20]
fig, axs = plt.subplots(2, 4, figsize=(18, 8))
for ax, a in zip(axs.flat, AG):
    px.shade_icmr_window(ax, label=False)
    for m in ps.METHODS:
        ax.plot(S[m].index, S[m][a], color=MCOL[m], ls=MSTY[m], lw=1.3, label=ps.METHOD_LABEL[m])
    px.style_pop_axis(ax, f"Age {a}", ymax=max(S[m][a].max() for m in ps.METHODS))
axs[0, 0].legend(frameon=False, fontsize=7)
plt.tight_layout(); px.save(fig, f"{TAG}_single_ages_time"); plt.show()

---
## 3. Custom age bands

| Scheme | Bands |
|---|---|
| S0 | The original 16 five-year bands (check: must equal the track) |
| S1 | 0–8, 9–14, 15–19, then the original five-year bands to 75+ |
| S2 | 0–17, 18–29, 30–65, 66+ |
| S3 | 0–8, 9–14, 15–26, 27–45, 46–65, 66+ |

Any other scheme can be added to `ps.SCHEMES` or passed directly to `ps.regroup(single_ages, [(label, first_age, last_age), ...])`.

In [ ]:
CUSTOM = {sch: {m: ps.regroup(S[m], bands_) for m in ps.METHODS} for sch, bands_ in ps.SCHEMES.items()}
s0 = max((CUSTOM["S0 original 5-year bands"][m].values / india.values - 1).max() for m in ps.METHODS)
print(f"S0 reproduces the track for every method: max relative difference {s0:.1e}")

### 3.1 Scheme S1 (0–8, 9–14, 15–19, …) over time

The first three bands of S1 by method; the original 00–04, 05–09 and 10–14 bands are shown for reference.

In [ ]:
sch = "S1 0-8, 9-14, 15-19, then 5-year"
fig, axs = plt.subplots(1, 4, figsize=(19, 4.5))
for ax, lab in zip(axs[:3], ["0-8", "9-14", "15-19"]):
    px.shade_icmr_window(ax, label=False)
    for m in ps.METHODS:
        ax.plot(india.index, CUSTOM[sch][m][lab], color=MCOL[m], ls=MSTY[m], lw=1.5, label=ps.METHOD_LABEL[m])
    px.style_pop_axis(ax, f"Band {lab}", ymax=CUSTOM[sch]["wpp_smooth"][lab].max())
ax = axs[3]
for b, c in zip(["00-04", "05-09", "10-14"], [sp.L1, sp.L2, sp.ACC]):
    ax.plot(india.index, india[b], color=c, lw=1.5, label=b)
px.style_pop_axis(ax, "Original bands (reference)", ymax=india[["00-04", "05-09", "10-14"]].values.max())
axs[0].legend(frameon=False, fontsize=7); ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); px.save(fig, f"{TAG}_S1_time"); plt.show()

rel = pd.DataFrame({(ps.METHOD_LABEL[m], yr): (CUSTOM[sch][m].loc[yr, ["0-8", "9-14"]] / CUSTOM[sch]["wpp_smooth"].loc[yr, ["0-8", "9-14"]] - 1) * 100
                    for m in ps.METHODS if m != "wpp_smooth" for yr in [1950, 2011, 2036, 2070, 2100]}).T.round(2)
print("Difference from M4 [%] for the S1 bands that cut through five-year bands:")
rel

### 3.2 Do custom bands introduce kinks over time?

For every band of every scheme: the largest year-to-year change in its annual growth rate (percentage points). The first row is the
same measure for the track's original 16 bands, the level of "kinkiness" already present before any splitting. Custom bands that
stay at or below that level have not gained new kinks from the split.

In [ ]:
base = ps.time_kink(india).max()
rows = {("original 16 bands", "(track)"): {"largest growth kink (pp/yr)": base, "band": ps.time_kink(india).idxmax()}}
for sch in list(ps.SCHEMES)[1:]:
    for m in ps.METHODS:
        k = ps.time_kink(CUSTOM[sch][m])
        rows[(sch, ps.METHOD_LABEL[m])] = {"largest growth kink (pp/yr)": k.max(), "band": k.idxmax()}
kinks = pd.DataFrame(rows).T
kinks["largest growth kink (pp/yr)"] = kinks["largest growth kink (pp/yr)"].astype(float).round(2)

fig, axs = plt.subplots(1, 3, figsize=(18, 4.6), sharey=True)
for ax, sch in zip(axs, list(ps.SCHEMES)[1:]):
    labs = [l for l, _, _ in ps.SCHEMES[sch]]
    x = np.arange(len(labs))
    for i, m in enumerate(ps.METHODS):
        ax.bar(x + (i - 1.5) * 0.2, ps.time_kink(CUSTOM[sch][m])[labs], width=0.2, color=MCOL[m], label=ps.METHOD_LABEL[m])
    ax.axhline(base, color="black", lw=1, ls="--", label="max for original bands")
    ax.set_xticks(x); ax.set_xticklabels(labs, rotation=60, ha="right", fontsize=7); sp.lock_ticks(ax, "x")
    ax.set_title(sch, fontsize=9); ax.set_ylabel("Largest growth kink [pp/yr]"); ax.grid(alpha=0.25, axis="y")
axs[0].legend(frameon=False, fontsize=7)
plt.tight_layout(); px.save(fig, f"{TAG}_custom_kinks"); plt.show()
kinks

### 3.3 Schemes S2 and S3 over time (M4, with M1–M3 as thin lines)

In [ ]:
for sch in ["S2 0-17, 18-29, 30-65, 66+", "S3 0-8, 9-14, 15-26, 27-45, 46-65, 66+"]:
    labs = [l for l, _, _ in ps.SCHEMES[sch]]
    fig, axs = plt.subplots(1, len(labs), figsize=(3.3 * len(labs), 3.9))
    for ax, lab in zip(axs, labs):
        px.shade_icmr_window(ax, label=False)
        for m in ps.METHODS:
            ax.plot(india.index, CUSTOM[sch][m][lab], color=MCOL[m], ls=MSTY[m], lw=1.8 if m == "wpp_smooth" else 1.0,
                    label=ps.METHOD_LABEL[m])
        px.style_pop_axis(ax, f"Band {lab}", ymax=CUSTOM[sch]["wpp_smooth"][lab].max())
    axs[0].legend(frameon=False, fontsize=6)
    fig.suptitle(sch, y=1.03)
    plt.tight_layout(); px.save(fig, f"{TAG}_{sch.split()[0]}_time"); plt.show()

---
## 4. States: how much does the method matter for a policy band?

The 9–14 band cuts through two five-year bands (05–09 and 10–14), so it depends on the split. Percent difference of M1, M2 and M3
from M4 for every state and year.

In [ ]:
def band_state(m, lo=9, hi=14):
    return pd.DataFrame({s: ps.split(states[s], m, W).loc[:, lo:hi].sum(axis=1) for s in STATES})
b914 = {m: band_state(m) for m in ps.METHODS}
fig, axs = plt.subplots(1, 3, figsize=(19, 9), sharey=True)
norm = TwoSlopeNorm(vmin=-6, vcenter=0, vmax=6)
for ax, m in zip(axs, ["uniform", "wpp", "cumspline"]):
    D = (b914[m] / b914["wpp_smooth"] - 1) * 100
    im = ax.imshow(D.T.values, aspect="auto", cmap="RdBu_r", norm=norm,
                   extent=[D.index.min() - 0.5, D.index.max() + 0.5, len(STATES) - 0.5, -0.5])
    ax.set_yticks(range(len(STATES))); ax.set_yticklabels(STATES, fontsize=7); sp.lock_ticks(ax, "y")
    ax.set_title(f"9–14: {ps.METHOD_LABEL[m]} vs M4 [%]"); ax.set_xlabel("Year")
fig.colorbar(im, ax=axs, shrink=0.8, label="Difference from M4 [%]")
px.save(fig, f"{TAG}_states_9_14"); plt.show()
summary_9_14 = pd.DataFrame({ps.METHOD_LABEL[m]: ((b914[m] / b914["wpp_smooth"] - 1) * 100).abs().stack().describe()[["mean", "50%", "max"]]
                             for m in ["uniform", "wpp", "cumspline"]}).T.round(2)
summary_9_14

In [ ]:
STATE_TO_PLOT = "Kerala"
fig, axs = plt.subplots(1, 2, figsize=(16, 4.5))
for ax, yr in zip(axs, [2011, 2050]):
    for m in ps.METHODS:
        ax.plot(ps.AGES, ps.split(states[STATE_TO_PLOT], m, W).loc[yr] / 1e3, color=MCOL[m], ls=MSTY[m], lw=1.4, label=ps.METHOD_LABEL[m])
    for lo_b, _ in ps.BAND_RANGE[1:]:
        ax.axvline(lo_b - 0.5, color=sp.GREY, lw=0.5, ls=":")
    ax.set_xlabel("Age [years]"); ax.set_ylabel("Females [thousands]"); ax.set_title(f"{STATE_TO_PLOT}, {yr}"); ax.grid(alpha=0.25)
axs[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); px.save(fig, f"{TAG}_state_profile"); plt.show()

---
## 5. Save outputs

Written to `outputs/single_age/<TAG>/`:
- `india_single_ages.xlsx` — one sheet per method (years × ages 0 … 99, 100+)
- `states_single_ages_<method>.parquet` — all states, long format (`state, year, age, females`)
- `india_custom_bands_<method>.csv` — every scheme in `ps.SCHEMES`, long format (`scheme, band, year, females`)

In [ ]:
OUT = os.path.join(pm.OUT_DIR, "single_age", TAG); os.makedirs(OUT, exist_ok=True)
with pd.ExcelWriter(os.path.join(OUT, "india_single_ages.xlsx")) as xw:
    for m in ps.METHODS:
        out = S[m].copy(); out.columns = ps.AGE_LABELS; out.index.name = "Year"; out.to_excel(xw, sheet_name=m)
for m in ps.METHODS:
    long = pd.concat({s: ps.split(states[s], m, W) for s in STATES}, names=["state", "year"])
    long.columns.name = "age"
    long.stack().rename("females").reset_index().to_parquet(os.path.join(OUT, f"states_single_ages_{m}.parquet"), index=False)
    pd.concat({sch.split()[0]: CUSTOM[sch][m] for sch in ps.SCHEMES}, names=["scheme", "year"]).rename_axis(columns="band") \
      .stack().rename("females").reset_index().to_csv(os.path.join(OUT, f"india_custom_bands_{m}.csv"), index=False)
print(OUT); print(sorted(os.listdir(OUT)))

### 6. Results (India, B-taper)

- **Band totals** are reproduced exactly by every method, for India and all states.
- **Benchmark (Section 1):** M4 is the most accurate below age 75 when the WPP shape is imperfect (MAPE 0.8–1.0 %); above 75 only M2 and M4 are usable (11–29 % vs 220–1050 % for M3/M1).
- **Band-edge steps:** M1 and M2 bend about 3× more at band edges than inside; M3 and M4 are smooth.
- **One-year cohort test:** M2 7 % of cells with cohort growth, M4 14 %, M3 26 %, M1 66 %. At band edges M4 is best (19 % vs 24 % for M2). M2 keeps WPP's within-band shape exactly (good cohort flow inside bands) but steps at the edges; M4 removes the steps at a small cost inside bands.
- **Custom bands over time:** no method adds kinks beyond the track's own (the largest, 5.2 pp/yr in 70–74 and 2.3 pp/yr in 66+, come from the track at 1991 / 2011). Track B's bands already follow WPP's dynamics, so WPP's single-year detail adds nothing new.
- **How much the method matters:** 0–8 and 9–14 differ by less than 1 % between methods at India level; across states, 9–14 differs from M4 by a median of 0.1 % (M3) to 0.4 % (M1, M2), at most 1.3–4.2 %.